In [1]:
!python -m pip install -U python-woc pandas matplotlib
# please clear the output of this cell in your notebook before checking it in

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 3.2 MB 5.2 MB/s            
     |████████████████████████████████| 12.8 MB 63.8 MB/s            
     |████████████████████████████████| 8.3 MB 58.4 MB/s            
     |████████████████████████████████| 80 kB 7.5 MB/s             
  Preparing metadata (setup.py) ... done
     |████████████████████████████████| 8.2 MB 65.2 MB/s            
     |████████████████████████████████| 199 kB 69.0 MB/s            
     |████████████████████████████████| 321 kB 89.0 MB/s            
     |████████████████████████████████| 4.8 MB 147.8 MB/s            
Using legacy 'setup.py install' for python-lzf, since package 'wheel' is not installed.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
    Runni

In [3]:
from woc.remote import WocMapsRemote
from tqdm import tqdm
import pandas as pd
# creates the client
woc = WocMapsRemote( base_url="https://worldofcode.org/api/")
# if you got an API key
# woc = WocMapsRemote( base_url="https://worldofcode.org/api/", api_key="woc-XXXXXX-YYYYYY" )

# Now for each of the ten projects you were assigned get commits, e.g.
projects = [
    'helske/KFAS',
    'sassoftware/python-swat',
    'spacetelescope/mirage',
    'devinamatthews/tblis',
    'esheldon/esutil',
    'badlands-model/badlands',
    'metagenome-atlas/atlas',
    'opengeospatial/teamengine',
    'philipperemy/keras-tcn',
    'libnegf/libnegf'
]
list_df_commits = []
for prj in projects:
  # Convert GitHub repository names to WoC V2412 project names
  prj = prj.lower().replace('/','_',2)
  commits = woc.get_values('p2c', prj)
  df = pd.DataFrame(commits, columns=["sha1"])
  df['project'] = prj
  list_df_commits.append(df)
df = pd.concat(list_df_commits)
#perhaps save the list (if no errors) so you do not need to retrieve them again
df.to_csv('df_commits.csv')
df.head(1)

,sha1,project
0,006fe5f3b076b0f9cc5c1284c7b10e880b5b206e,helske_kfas


In [4]:
df.groupby('project').size()

project
badlands-model_badlands       615
devinamatthews_tblis          636
esheldon_esutil              1499
helske_kfas                   393
libnegf_libnegf               923
metagenome-atlas_atlas       3834
opengeospatial_teamengine    2739
philipperemy_keras-tcn        651
sassoftware_python-swat       760
spacetelescope_mirage        2525
dtype: int64

In [7]:
# If the number of commits its not very large, you can try to get them all at the same time,
# The max batch size is 10
import time
# let us first split df['sha1'] in chunks
chunks = [df['sha1'][x:x+10] for x in range(0, len(df), 10)]
commit_data = []

for chunk in tqdm(chunks): # iterate over the commits
  # res, err = woc.show_content_many('commit',chunk.to_list())

  # commit.tch returns the same data but faster
  res, err = woc.get_values_many('commit.tch',chunk.to_list())
  res = {k: v[0] for k, v in res.items()}  # this conversion is necessary because of the internal implementation

  # to walk around the rate limit
  time.sleep(1)

  if err: # check for errors
    print('Got Errors', err)

  for commit_sha, commit in res.items():
    # flatten commit objects
    commit_data.append({
          'commit': commit_sha,
          'tree': commit[0],
          'parent': list(commit[1]),
          'author': commit[2][0],
          'author_time': int(commit[2][1]),
          'author_tz': commit[2][2],
          'committer': commit[3][0],
          'committer_time': int(commit[3][1]),
          'committer_tz': commit[3][2],
          'message': commit[4],
    })

df_commit_data = pd.DataFrame(commit_data)
df_commit_data = df_commit_data.merge(df, left_on='commit', right_on='sha1')
df_commit_data.to_csv('df_commit_data.csv', index=False)
df_commit_data.head(2)

  6%|████▊                                                                           | 87/1458 [01:30<24:15,  1.06s/it]

Got Errors {'9d7692c51b1509281b998300d70d9159ef076035': 'Key 9d7692c51b1509281b998300d70d9159ef076035 not found in /da5_fast/All.sha1c/commit_29.tch'}


  8%|██████                                                                         | 112/1458 [01:57<23:27,  1.05s/it]

Got Errors {'f10c32ffa0b033ca0342b6c4aec0a536722b4512': 'Key f10c32ffa0b033ca0342b6c4aec0a536722b4512 not found in /da5_fast/All.sha1c/commit_113.tch'}


 11%|████████▊                                                                      | 163/1458 [02:50<23:00,  1.07s/it]

...

Got Errors {'23605f62ba14d35eb67bb126f8989155fd323491': 'Key 23605f62ba14d35eb67bb126f8989155fd323491 not found in /da5_fast/All.sha1c/commit_35.tch'}


 99%|█████████████████████████████████████████████████████████████████████████████▏| 1442/1458 [25:01<00:16,  1.05s/it]

Got Errors {'d3c6d1f9b131f673fa0999ded56dd9b05847f14a': 'Key d3c6d1f9b131f673fa0999ded56dd9b05847f14a not found in /da5_fast/All.sha1c/commit_83.tch'}


100%|██████████████████████████████████████████████████████████████████████████████| 1458/1458 [25:18<00:00,  1.04s/it]


,commit,tree,parent,author,author_time,author_tz,committer,committer_time,committer_tz,message,sha1,project
0,006fe5f3b076b0f9cc5c1284c7b10e880b5b206e,65410606c5b9411d7657298dff157dd75d10ca7a,[7abea1457c3b7dde0e4d6191a3b0927862c712cc],Jouni Helske <jouni.helske@jyu.fi>,1421925972,+0200,Jouni Helske <jouni.helske@jyu.fi>,1421925972,+0200,"Recursive residuals for non-Gaussian models, c...",006fe5f3b076b0f9cc5c1284c7b10e880b5b206e,helske_kfas
1,011a6a725b4f1b64d8faed510257e963aac90831,cfddc9f7fedb95ae1f3e58af5fc6da1c1338263a,[16019ebfb48d5a7e25bfe9efd7a0ea256489bd31],Jouni Helske <jvhels@utu.fi>,1747651599,+0300,Jouni Helske <jvhels@utu.fi>,1747651599,+0300,...\n,011a6a725b4f1b64d8faed510257e963aac90831,helske_kfas


In [9]:
# what does the first record looks like?
# commit  parents
v = commit_data[0]
# commit tree, [parent{s}], [author, unixtime, tz ], [ committer, unixtime, tz ], Commit_message ]
print(v)

{'commit': '006fe5f3b076b0f9cc5c1284c7b10e880b5b206e', 'tree': '65410606c5b9411d7657298dff157dd75d10ca7a', 'parent': ['7abea1457c3b7dde0e4d6191a3b0927862c712cc'], 'author': 'Jouni Helske <jouni.helske@jyu.fi>', 'author_time': 1421925972, 'author_tz': '+0200', 'committer': 'Jouni Helske <jouni.helske@jyu.fi>', 'committer_time': 1421925972, 'committer_tz': '+0200', 'message': 'Recursive residuals for non-Gaussian models, corrected few small bugs from\nprevious commit, formatted code for publication\n'}


In [10]:
# Create the final Part 1 summary file

summary_df = df_commit_data[
    ['project', 'commit', 'author', 'author_time', 'message']
].copy()

# Rename columns to match the assignment requirements
summary_df.columns = [
    'project_wocid',
    'commit_sha1',
    'author',
    'time',
    'commit message'
]

# Save as a semicolon-separated CSV
summary_df.to_csv(
    'bnoyes_project_summary.csv',
    sep=';',
    index=False
)

# Check the result
print("Total rows:", len(summary_df))
print("Projects:", summary_df['project_wocid'].nunique())

summary_df.head()

Total rows: 14497
Projects: 10


,project_wocid,commit_sha1,author,time,commit message
0,helske_kfas,006fe5f3b076b0f9cc5c1284c7b10e880b5b206e,Jouni Helske <jouni.helske@jyu.fi>,1421925972,"Recursive residuals for non-Gaussian models, c..."
1,helske_kfas,011a6a725b4f1b64d8faed510257e963aac90831,Jouni Helske <jvhels@utu.fi>,1747651599,...\n
2,helske_kfas,0264561e8f03eced1651f805518c8d69a1c388ff,Jouni Helske <jouni.helske@jyu.fi>,1465661996,updated Changelog\n
3,helske_kfas,026e061ac98484834c446155e43f49cecc294817,Jouni Helske <jouni.helske@jyu.fi>,1435237762,minor typo\n
4,helske_kfas,02c08325dc89c4cddf87e80093bc1aa64160c72c,helske <jouni.helske@jyu.fi>,1453204357,corrected expected output\n


In [11]:
# Calculate WoC statistics for each project

project_stats = summary_df.groupby('project_wocid').agg(
    ncommits=('commit_sha1', 'count'),
    nauthors=('author', 'nunique'),
    min_time=('time', 'min'),
    max_time=('time', 'max')
).reset_index()

# Convert Unix timestamps to readable dates
project_stats['min_time'] = pd.to_datetime(
    project_stats['min_time'],
    unit='s'
)

project_stats['max_time'] = pd.to_datetime(
    project_stats['max_time'],
    unit='s'
)

project_stats

,project_wocid,ncommits,nauthors,min_time,max_time
0,badlands-model_badlands,615,19,2015-03-30 23:41:22,2026-03-10 03:54:52
1,devinamatthews_tblis,634,26,2015-09-30 16:44:25,2025-10-09 04:17:09
2,esheldon_esutil,1484,26,2009-09-17 19:20:08,2025-05-03 11:20:50
3,helske_kfas,393,10,2014-04-04 13:37:52,2025-05-25 20:00:20
4,libnegf_libnegf,921,41,2010-06-07 11:39:09,2025-10-08 13:42:40
5,metagenome-atlas_atlas,3833,62,2016-06-02 23:18:00,2025-12-03 14:30:31
6,opengeospatial_teamengine,2698,72,2006-07-26 21:50:57,2025-10-28 19:29:23
7,philipperemy_keras-tcn,648,41,2018-03-22 02:40:16,2025-04-08 04:11:42
8,sassoftware_python-swat,758,29,2016-07-11 19:38:40,2026-04-08 12:41:27
9,spacetelescope_mirage,2513,38,2017-11-08 21:36:42,2025-10-27 18:48:32


In [ ]:
# Now that the data is retrieved, save it and commit to your GH fork
# First, we need to flatten info in order to export as csv
# our dataframe will have columns project,'commit, author, time, message'
dfinf = pd.DataFrame(columns=['project', 'commit', 'author', 'time', 'message'])
for k in commit_data:
  row = pd.Series({'project':prj, 'commit': k['commit'], 'author': k['author'], 'time': k['author_time'], 'message':k['message']})
  dfinf = pd.concat([dfinf, row.to_frame().T ], ignore_index=True)


In [ ]:
# check if it has the right content
dfinf.head(1)

In [ ]:
#mode a means append, so you have all your projects in the same file
yournetid='your_net_id_here'
dfinf.to_csv(yournetid+'_project_summary.csv', index=False,sep=';', mode='a', header=False)

# Make sure you check in to your fork not just the notebook but also the csv files!!!

# Don't forget to add requested data from github and this notebook
### For each of the 10 projects go to their github repo and get the number of stars, number of forks, and the last commit date
### Report (in your notebook) the number of commits, the number of authors, and max and min time for each project based on WoC commits and also add the info you obtained from github

In [ ]:
## helske_kfas
- Number of Stars: 73
- Number of Forks: 15
- Last Commit Date: May 25, 2025

## sassoftware_python-swat
- Number of Stars: 159
- Number of Forks: 65
- Last Commit Date: June 29, 2026

## spacetelescope_mirage
- Number of Stars: 46
- Number of Forks: 41
- Last Commit Date: July 16, 2026

## devinamatthews_tblis
- Number of Stars: 145
- Number of Forks: 39
- Last Commit Date: November 21, 2025

## esheldon_esutil
- Number of Stars: 35
- Number of Forks: 21
- Last Commit Date: September 14, 2026

## badlands-model_badlands
- Number of Stars: 192
- Number of Forks: 50
- Last Commit Date: April 30, 2026

## metagenome-atlas_atlas
- Number of Stars: 406
- Number of Forks: 96
- Last Commit Date: March 19, 2025

## opengeospatial_teamengine
- Number of Stars: 57
- Number of Forks: 41
- Last Commit Date: March 17, 2025

## philipperemy_keras-tcn
- Number of Stars: 2K
- Number of Forks: 463
- Last Commit Date: June 16, 2026

## libnegf_libnegf
- Number of Stars: 26
- Number of Forks: 14
- Last Commit Date: July 16, 2025

In [ ]:
## World of Code Project Information

## badlands-model_badlands
- Number of Commits: 615
- Number of Authors: 19
- Min Time: 2015-03-30 23:41:22
- Max Time: 2026-03-10 03:54:52

## devinamatthews_tblis
- Number of Commits: 634
- Number of Authors: 26
- Min Time: 2015-09-30 16:44:25
- Max Time: 2025-10-09 04:17:09

## esheldon_esutil
- Number of Commits: 1484
- Number of Authors: 26
- Min Time: 2009-09-17 19:20:08
- Max Time: 2025-05-03 11:20:50

## helske_kfas
- Number of Commits: 393
- Number of Authors: 10
- Min Time: 2014-04-04 13:37:52
- Max Time: 2025-05-25 20:00:20

## libnegf_libnegf
- Number of Commits: 921
- Number of Authors: 41
- Min Time: 2010-06-07 11:39:09
- Max Time: 2025-10-08 13:42:40

## metagenome-atlas_atlas
- Number of Commits: 3833
- Number of Authors: 62
- Min Time: 2016-06-02 23:18:00
- Max Time: 2025-12-03 14:30:31

## opengeospatial_teamengine
- Number of Commits: 2698
- Number of Authors: 72
- Min Time: 2006-07-26 21:50:57
- Max Time: 2025-10-28 19:29:23

## philipperemy_keras-tcn
- Number of Commits: 648
- Number of Authors: 41
- Min Time: 2018-03-22 02:40:16
- Max Time: 2025-04-08 04:11:42

## sassoftware_python-swat
- Number of Commits: 758
- Number of Authors: 29
- Min Time: 2016-07-11 19:38:40
- Max Time: 2026-04-08 12:41:27

## spacetelescope_mirage
- Number of Commits: 2513
- Number of Authors: 38
- Min Time: 2017-11-08 21:36:42
- Max Time: 2025-10-27 18:48:32